# Building Production-Ready ML Pipelines with scikit-learn

**Audience:** mixed backgrounds · **Duration:** 2.5 hours · **Dataset:** Titanic-like passenger data generated locally

This notebook moves from a simple baseline to a reusable, leak-free workflow. Run cells from top to bottom.

## Learning path

1. Meet the data and define the prediction problem  
2. Build a simple baseline  
3. See how leakage happens  
4. Preprocess numbers and categories  
5. Combine everything in a Pipeline  
6. Evaluate with cross-validation  
7. Tune, save and reload  
8. Discuss production guardrails

In [ ]:
# Run this first
import sys, subprocess, importlib.util
for package in ["numpy", "pandas", "sklearn", "joblib", "matplotlib"]:
    name = "scikit-learn" if package == "sklearn" else package
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score

RANDOM_STATE = 42
print("Ready")

## 1. Create a small, realistic dataset

The notebook generates data locally, so it works without internet. The target is `survived`.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 900
df = pd.DataFrame({
    "age": np.clip(rng.normal(30, 14, n), 1, 80),
    "fare": np.round(rng.lognormal(3.1, 0.75, n), 2),
    "siblings_spouses": rng.integers(0, 5, n),
    "parents_children": rng.integers(0, 4, n),
    "sex": rng.choice(["female", "male"], n),
    "passenger_class": rng.choice(["First", "Second", "Third"], n, p=[.2, .25, .55]),
    "embarked": rng.choice(["C", "Q", "S"], n, p=[.2, .1, .7]),
})

# Add missing values
for col, rate in {"age": .12, "fare": .03, "embarked": .04}.items():
    df.loc[rng.random(n) < rate, col] = np.nan

# Create a learnable target with noise (for teaching only)
age_for_target = df["age"].fillna(30)
logit = (
    -0.8
    + 1.4 * (df["sex"] == "female")
    + 0.9 * (df["passenger_class"] == "First")
    + 0.35 * (df["passenger_class"] == "Second")
    - 0.018 * age_for_target
    + rng.normal(0, 0.7, n)
)
probability = 1 / (1 + np.exp(-logit))
df["survived"] = rng.binomial(1, probability)
df.head()

In [ ]:
df.info()
print("\nTarget balance:")
display(df["survived"].value_counts(normalize=True).rename("proportion"))

### Pause and discuss

- Which columns are numerical?
- Which are categorical?
- Which columns contain missing values?
- What must happen to a new row before a model can use it?

## 2. Split before learning anything from the data

The test set represents future, unseen data. We protect it until final evaluation.

In [ ]:
X = df.drop(columns="survived")
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
print(X_train.shape, X_test.shape)

## 3. What leakage looks like

**Do not copy this pattern:** fitting preprocessing before the split lets test-set information influence training.

```python
scaler.fit(X_all)  # test rows influence mean and standard deviation
X_train, X_test = train_test_split(X_all)
```

The safe rule is: **split first; fit learned transformations on training data only.**

## 4. Step-by-step production problems

We will fix one problem at a time, then combine the fixes into one pipeline.

### 4.1 Missing values

Never calculate an imputation value from the full dataset. The value must be learned from training rows only.

In [ ]:
print("Missing values before preprocessing:")
display(X_train.isna().sum())

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(X_train[["age"]])
print("Training-only age median:", age_imputer.statistics_[0])

### 4.2 Scaling

Scaling is useful for many linear and distance-based models. The scaler learns the training distribution and reuses it later.

In [ ]:
scaler = StandardScaler()
scaled_age = scaler.fit_transform(X_train[["age"]].fillna(X_train["age"].median()))
print("Scaled training age mean:", round(scaled_age.mean(), 6))

### 4.3 Outliers

Outlier treatment is a policy decision. First inspect it; do not silently delete values in a production workflow.

In [ ]:
print(X_train["fare"].describe(percentiles=[.01, .5, .99]))
fare_cap = X_train["fare"].quantile(.99)
print("Example training-only cap:", round(fare_cap, 2))

### 4.4 Reproducible transformations

If the notebook creates a transformed feature such as log1p(fare), the serving path must apply the same transformation. In this lesson we keep the transformation inside the preprocessing design.

In [ ]:
fare_log = np.log1p(X_train[["fare"]].fillna(X_train["fare"].median()))
print(fare_log.head())

### 4.5 Categorical treatment

One-hot encoding turns categories into numeric columns. The unknown-category setting protects inference when a new category appears.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(X_train[["sex", "passenger_class", "embarked"]].fillna("Unknown"))
print("Encoded feature count:", encoded.shape[1])

## 5. Combine all fixes in a pipeline

In [ ]:
numeric_features = ["age", "fare", "siblings_spouses", "parents_children"]
categorical_features = ["sex", "passenger_class", "embarked"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
preprocessor

### Why `handle_unknown="ignore"`?

Production data may contain a category not observed during training. Ignoring it prevents inference from crashing.

## 6. Create one end-to-end pipeline

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

pipeline.fit(X_train, y_train)
test_predictions = pipeline.predict(X_test)
test_probabilities = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, test_predictions))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_probabilities), 3))
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions);

### Key idea

The pipeline accepts **raw rows**. It performs imputation, encoding, scaling and prediction in the correct order.

## 7. Evaluate the whole workflow with cross-validation

In [ ]:
cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc",
)
print("Fold scores:", np.round(cv_scores, 3))
print(f"Mean ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

Each fold fits its own imputer, encoder, scaler and model. That is why keeping preprocessing inside the pipeline matters.

## 8. Tune a parameter safely

In [ ]:
parameter_grid = {
    "model__C": [0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

search = GridSearchCV(
    pipeline,
    parameter_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
)
search.fit(X_train, y_train)
print("Best settings:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 3))

The double underscore in `model__C` means “parameter `C` inside the step named `model`.”

## 9. Save and reload the complete workflow

In [ ]:
model_path = "titanic_pipeline.joblib"
joblib.dump(search.best_estimator_, model_path)
loaded_pipeline = joblib.load(model_path)

new_passengers = pd.DataFrame([{
    "age": 29,
    "fare": 85.0,
    "siblings_spouses": 0,
    "parents_children": 0,
    "sex": "female",
    "passenger_class": "First",
    "embarked": "C",
}])

print("Prediction:", loaded_pipeline.predict(new_passengers)[0])
print("Probability:", round(loaded_pipeline.predict_proba(new_passengers)[0, 1], 3))

## 10. Student exercise

1. Change numerical imputation from median to mean.
2. Add `class_weight="balanced"` directly to the classifier.
3. Compare cross-validation scores.
4. Create a new raw passenger with one missing value.
5. Confirm that the saved pipeline can still predict.

**Challenge:** replace logistic regression with a random forest without rewriting preprocessing.

In [ ]:
# TODO: Try your changes here


## 11. Production readiness checklist

- Validate column names, types and allowed ranges.
- Pin Python and library versions.
- Test training and inference paths.
- Record model and dataset versions.
- Log latency and failures without exposing sensitive data.
- Monitor input drift and prediction quality.
- Define when and how retraining happens.

> Saving a pipeline is the start of production readiness—not the whole of MLOps.

## 12. Define the inference contract

The service accepts a raw passenger record and returns a probability, decision and model version. In a real service, validate this schema before calling the pipeline.

```json
{
  "age": 29,
  "fare": 85.0,
  "siblings_spouses": 0,
  "parents_children": 0,
  "sex": "female",
  "passenger_class": "First",
  "embarked": "C"
}
```

In [ ]:
def predict_contract(raw_record, model, model_version="1.0"):
    required = set(X_train.columns)
    missing = required - set(raw_record)
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    probability = float(model.predict_proba(pd.DataFrame([raw_record]))[0, 1])
    return {
        "probability": probability,
        "decision": int(probability >= 0.5),
        "model_version": model_version,
    }

predict_contract(new_passengers.iloc[0].to_dict(), loaded_pipeline)

## 13. Release tests

Before deployment, test the contract—not only the estimator.

- Unit test: missing values and unknown categories.
- Integration test: raw request to documented response.
- Regression test: known records remain stable.
- Load test: latency remains acceptable under traffic.
- Rollback test: the previous artifact can be restored.

In [ ]:
valid_record = new_passengers.iloc[0].to_dict()
response = predict_contract(valid_record, loaded_pipeline, "1.0")

assert 0.0 <= response["probability"] <= 1.0
assert response["decision"] in [0, 1]
assert response["model_version"] == "1.0"
print("Contract test passed:", response)

## 14. Monitor after deployment

Track two different layers:

**Operational signals:** request volume, latency, error rate, invalid schemas and resource use.

**ML signals:** missing-value rates, category frequencies, feature drift, prediction distribution and—when labels arrive—real model quality.

A drift alert is a reason to investigate. It is not automatically a reason to retrain.

In [ ]:
training_reference = X_train["age"].median()
production_batch = pd.DataFrame({"age": [42, 48, 51, 46, 55]})
production_median = production_batch["age"].median()

drift_signal = abs(production_median - training_reference)
print({
    "training_age_median": round(training_reference, 1),
    "production_age_median": round(production_median, 1),
    "simple_shift_signal": round(drift_signal, 1),
})

## 15. Close the retraining loop

1. Detect a quality, drift or business signal.
2. Investigate whether it is real or a data/logging problem.
3. Build a candidate with approved, reproducible data.
4. Compare it with the production model using agreed tests.
5. Release gradually using shadow or canary traffic.
6. Promote the candidate or roll back.

The production model is a continuously operated system, not a file copied from a notebook.

## Final takeaway

**Preprocessing + model = the real model.**

Split first. Put every learned transformation inside the pipeline. Evaluate the complete workflow. Save and serve it as one unit.